# FULL Benchmark Accuracy Evaluation — Enterprise BFSI Client MDM

**Purpose:** Evaluate the already-produced FULL Databricks outputs against the benchmark's hidden `entity_key` ground truth. This notebook is read-only against the operational tables; it does not rerun or modify matching, stewardship, Gold, or source data.

**Important:** The ground-truth mapping is used ONLY in this separate evaluation notebook. Never add `entity_key` to Silver, the blocking features, `mdm.match_results`, or production/serving tables.

## Before running
1. Locate the exact FULL `ground_truth.csv` that corresponds to the benchmark run (normally `source_system,source_id,entity_key`, with 2,638,899 mapping rows).
2. Upload a copy to a restricted evaluation-only folder in the same ADLS Gen2 account or a governed Unity Catalog Volume that this workspace can read. Example ADLS URI shape: `abfss://<container>@<storage-account>.dfs.core.windows.net/<restricted-folder>/ground_truth.csv`.
3. **Do not** place the ground truth under the normal `source/` or `raw/` paths, and do not make it an input to any production pipeline.
4. Set the `GROUND_TRUTH_PATH` widget to the exact URI before running all cells.
5. Confirm the tables below still represent the same completed FULL run described by the project evidence.

## Metrics reported
- **Blocking recall:** how many true record-pairs survived candidate generation.
- **Pairwise automatic-match precision, recall and F1:** with `MATCH` treated as an accepted match; `REVIEW` is still unresolved and therefore does not count as an automatic true positive.
- **Final cluster pairwise precision, recall and F1:** whether records assigned to the same `MASTER_CLIENT_ID` belong to the same hidden entity.
- **Over-merged master clusters / split underlying entities:** direct checks for incorrect merges and incomplete merges.

The evaluation denominator uses the **trusted Silver population**, because contract/DQ-quarantined records were not eligible for the recorded matching run. The notebook reports ground-truth rows outside Silver separately.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

# ============================================================
# CONFIGURATION
# ============================================================

CATALOG = "bfsi_mdm_dev"

SILVER_TABLE = f"{CATALOG}.silver.all_clients"
CANDIDATE_TABLE = f"{CATALOG}.mdm.candidate_pairs"
MATCH_RESULTS_TABLE = f"{CATALOG}.mdm.match_results"
SOURCE_MAP_TABLE = f"{CATALOG}.gold.client_source_map"

# Supply the restricted FULL ground-truth path as a widget.
# The hidden entity_key is used only in this evaluation notebook.
dbutils.widgets.text(
    "GROUND_TRUTH_PATH",
    "",
    "FULL ground_truth.csv URI (restricted ADLS Gen2 path or Unity Catalog Volume)",
)
dbutils.widgets.text(
    "RESULT_OUTPUT_PATH",
    "",
    "Optional evaluation-only output folder URI (leave blank to only display)",
)

GROUND_TRUTH_PATH = dbutils.widgets.get("GROUND_TRUTH_PATH").strip()
RESULT_OUTPUT_PATH = dbutils.widgets.get("RESULT_OUTPUT_PATH").strip()

if not GROUND_TRUTH_PATH:
    raise ValueError(
        "Set the GROUND_TRUTH_PATH widget to the exact URI of the FULL "
        "ground_truth.csv file before running this notebook."
    )

print("Catalog:", CATALOG)
print("Silver:", SILVER_TABLE)
print("Candidate pairs:", CANDIDATE_TABLE)
print("Match results:", MATCH_RESULTS_TABLE)
print("Gold source map:", SOURCE_MAP_TABLE)
print("Ground-truth path configured: yes (path not printed)")
print("Output report:", "display only" if not RESULT_OUTPUT_PATH else "configured")

# Ground truth is used only for evaluation, never in the matching or serving pipeline.


Catalog: bfsi_mdm_dev
Silver: bfsi_mdm_dev.silver.all_clients
Candidate pairs: bfsi_mdm_dev.mdm.candidate_pairs
Match results: bfsi_mdm_dev.mdm.match_results
Gold source map: bfsi_mdm_dev.gold.client_source_map
Ground-truth path configured: yes
Output report: display only


## 1. Load and validate the ground truth and eligible Silver population

The evaluator joins on `(source_system, source_id)`. The hidden `entity_key` remains in evaluation-only data.

In [0]:
gt_raw = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(GROUND_TRUTH_PATH)
)

required_gt_columns = {"source_system", "source_id", "entity_key"}
missing_columns = required_gt_columns - set(gt_raw.columns)
if missing_columns:
    raise ValueError(f"Ground-truth file is missing columns: {sorted(missing_columns)}")

gt_normalized = gt_raw.select(
    F.upper(F.trim(F.col("source_system").cast("string"))).alias("source_system"),
    F.trim(F.col("source_id").cast("string")).alias("source_id"),
    F.trim(F.col("entity_key").cast("string")).alias("entity_key"),
)

gt_rows = gt_normalized.count()
invalid_gt_rows = gt_normalized.filter(
    F.col("source_system").isNull() | (F.col("source_system") == "")
    | F.col("source_id").isNull() | (F.col("source_id") == "")
    | F.col("entity_key").isNull() | (F.col("entity_key") == "")
).count()
if invalid_gt_rows:
    raise ValueError(
        f"Ground truth contains {invalid_gt_rows:,} rows with blank/null source_system, source_id, or entity_key. "
        "Do not silently discard ground-truth mappings."
    )
gt = gt_normalized

duplicate_gt_keys = (
    gt.groupBy("source_system", "source_id")
      .count()
      .filter(F.col("count") > 1)
      .limit(1)
      .count()
)
if duplicate_gt_keys:
    raise ValueError(
        "Ground truth contains duplicate (source_system, source_id) mappings. "
        "Do not calculate metrics until the mapping is corrected."
    )

silver_keys = (
    spark.table(SILVER_TABLE)
    .select(
        F.upper(F.trim(F.col("source_system").cast("string"))).alias("source_system"),
        F.trim(F.col("source_id").cast("string")).alias("source_id"),
    )
    .withColumn(
        "record_key",
        F.concat_ws("::", F.col("source_system"), F.col("source_id")),
    )
)

silver_rows = silver_keys.count()
duplicate_silver_keys = (
    silver_keys.groupBy("source_system", "source_id")
               .count()
               .filter(F.col("count") > 1)
               .limit(1)
               .count()
)
if duplicate_silver_keys:
    raise ValueError(
        "Silver contains duplicate (source_system, source_id) keys; "
        "ground-truth mapping would be ambiguous."
    )

silver_without_truth = (
    silver_keys.select("source_system", "source_id")
    .join(gt.select("source_system", "source_id"),
          ["source_system", "source_id"], "left_anti")
    .count()
)
truth_outside_silver = (
    gt.select("source_system", "source_id", "entity_key")
      .join(silver_keys.select("source_system", "source_id"),
            ["source_system", "source_id"], "left_anti")
      .count()
)

if silver_without_truth != 0:
    raise ValueError(
        f"{silver_without_truth:,} trusted Silver records have no ground-truth mapping. "
        "Verify you provided the ground truth for this exact FULL benchmark."
    )

truth_silver = (
    silver_keys
    .join(gt, ["source_system", "source_id"], "inner")
    .select("source_system", "source_id", "record_key", "entity_key")
)
eligible_rows = truth_silver.count()
eligible_entities = truth_silver.select("entity_key").distinct().count()

print(f"Ground-truth mapping rows loaded       : {gt_rows:,}")
print(f"Invalid/blank ground-truth rows         : {invalid_gt_rows:,}")
print(f"Trusted Silver records                 : {silver_rows:,}")
print(f"Silver records missing ground truth    : {silver_without_truth:,}")
print(f"Ground-truth records outside Silver    : {truth_outside_silver:,}")
print(f"Eligible records for accuracy scoring  : {eligible_rows:,}")
print(f"Underlying entities represented in Silver: {eligible_entities:,}")

# The eligible population should match trusted Silver exactly.
if eligible_rows != silver_rows:
    raise ValueError(
        f"Coverage reconciliation failed: eligible={eligible_rows:,}, Silver={silver_rows:,}."
    )

Ground-truth mapping rows loaded       : 2,638,899
Invalid/blank ground-truth rows         : 0
Trusted Silver records                 : 2,638,869
Silver records missing ground truth    : 0
Ground-truth records outside Silver    : 30
Eligible records for accuracy scoring  : 2,638,869
Underlying entities represented in Silver: 994,420


## 2. Calculate the true-pair denominator without materializing all pairs

For each hidden entity with `n` eligible source records, there are `n × (n-1) / 2` true record pairs. Summing this formula is equivalent to enumerating every true pair but is much cheaper.

In [0]:
entity_sizes = truth_silver.groupBy("entity_key").agg(F.count(F.lit(1)).alias("n"))
true_pair_expr = (
    (F.col("n").cast("long") * (F.col("n").cast("long") - F.lit(1)))
    / F.lit(2)
).cast("long")
true_pairs_total = entity_sizes.agg(F.sum(true_pair_expr).alias("true_pairs_total")).first()["true_pairs_total"]
true_pairs_total = int(true_pairs_total or 0)

print(f"Total true record pairs among trusted Silver: {true_pairs_total:,}")
if true_pairs_total <= 0:
    raise ValueError("No true record pairs found; check the selected ground truth and Silver population.")

record_truth = truth_silver.select("record_key", "entity_key")
truth_a = record_truth.select(
    F.col("record_key").alias("record_key_a"),
    F.col("entity_key").alias("entity_key_a"),
)
truth_b = record_truth.select(
    F.col("record_key").alias("record_key_b"),
    F.col("entity_key").alias("entity_key_b"),
)

Total true record pairs among trusted Silver: 2,508,438


## 3. Evaluate candidate blocking recall

The candidate table contains only record-key pairs. This section checks whether the pair shares a hidden `entity_key` with the other record. The ground truth is joined only here, in the evaluation notebook.

In [0]:
candidate_pairs = spark.table(CANDIDATE_TABLE).select(
    F.col("record_key_a").cast("string").alias("record_key_a"),
    F.col("record_key_b").cast("string").alias("record_key_b"),
)

candidate_labeled = (
    candidate_pairs
    .join(truth_a, "record_key_a", "left")
    .join(truth_b, "record_key_b", "left")
)

candidate_agg = candidate_labeled.agg(
    F.count(F.lit(1)).alias("candidate_rows"),
    F.sum(
        F.when(F.col("entity_key_a").isNull() | F.col("entity_key_b").isNull(), 1).otherwise(0)
    ).alias("candidate_rows_missing_truth"),
    F.sum(
        F.when(F.col("entity_key_a") == F.col("entity_key_b"), 1).otherwise(0)
    ).alias("true_candidate_pairs"),
).first()

candidate_rows = int(candidate_agg["candidate_rows"] or 0)
candidate_missing_truth = int(candidate_agg["candidate_rows_missing_truth"] or 0)
true_candidate_pairs = int(candidate_agg["true_candidate_pairs"] or 0)

if candidate_missing_truth:
    raise ValueError(
        f"{candidate_missing_truth:,} candidate-pair endpoints did not map to ground truth. "
        "Verify the candidate table and ground-truth benchmark version."
    )

blocking_recall = true_candidate_pairs / true_pairs_total
print(f"Candidate pair rows                    : {candidate_rows:,}")
print(f"True pairs present in candidates        : {true_candidate_pairs:,}")
print(f"True pairs missed by blocking           : {true_pairs_total - true_candidate_pairs:,}")
print(f"FULL blocking recall                   : {blocking_recall * 100:.4f}%")

Candidate pair rows                    : 138,074,372
True pairs present in candidates        : 2,495,317
True pairs missed by blocking           : 13,121
FULL blocking recall                   : 99.4769%


## 4. Evaluate the recorded direct matching decisions

`MATCH` is considered an automatic accepted edge. Correct true pairs in `REVIEW` are reported as unresolved; they are not counted as automatic true positives. The recall denominator includes all possible true pairs in the trusted Silver population, so it captures both blocking misses and candidates not automatically matched.

In [0]:
match_results = spark.table(MATCH_RESULTS_TABLE).select(
    F.col("record_key_a").cast("string").alias("record_key_a"),
    F.col("record_key_b").cast("string").alias("record_key_b"),
    F.upper(F.trim(F.col("decision").cast("string"))).alias("decision"),
)

match_labeled = (
    match_results
    .join(truth_a, "record_key_a", "left")
    .join(truth_b, "record_key_b", "left")
)

is_true_pair = F.col("entity_key_a") == F.col("entity_key_b")
match_agg = match_labeled.agg(
    F.count(F.lit(1)).alias("result_rows"),
    F.sum(
        F.when(F.col("entity_key_a").isNull() | F.col("entity_key_b").isNull(), 1).otherwise(0)
    ).alias("result_rows_missing_truth"),
    F.sum(F.when((F.col("decision") == "MATCH") & is_true_pair, 1).otherwise(0)).alias("tp"),
    F.sum(F.when((F.col("decision") == "MATCH") & (~is_true_pair), 1).otherwise(0)).alias("fp"),
    F.sum(F.when((F.col("decision") == "REVIEW") & is_true_pair, 1).otherwise(0)).alias("true_review_pairs"),
    F.sum(F.when((F.col("decision") == "NO_MATCH") & is_true_pair, 1).otherwise(0)).alias("true_no_match_pairs"),
    F.sum(F.when(~F.col("decision").isin("MATCH", "REVIEW", "NO_MATCH"), 1).otherwise(0)).alias("unexpected_decision_rows"),
).first()

match_rows = int(match_agg["result_rows"] or 0)
match_missing_truth = int(match_agg["result_rows_missing_truth"] or 0)
tp = int(match_agg["tp"] or 0)
fp = int(match_agg["fp"] or 0)
true_review_pairs = int(match_agg["true_review_pairs"] or 0)
true_no_match_pairs = int(match_agg["true_no_match_pairs"] or 0)
unexpected_decisions = int(match_agg["unexpected_decision_rows"] or 0)

if match_missing_truth:
    raise ValueError(
        f"{match_missing_truth:,} match-result endpoints did not map to ground truth."
    )
if unexpected_decisions:
    raise ValueError(
        f"{unexpected_decisions:,} rows have an unexpected decision label. Review before reporting accuracy."
    )
if match_rows != candidate_rows:
    raise ValueError(
        f"Candidate rows ({candidate_rows:,}) != match result rows ({match_rows:,}). "
        "The tables may be from different runs or one output may be incomplete; do not report metrics yet."
    )

fn = true_pairs_total - tp
precision = tp / (tp + fp) if (tp + fp) else 0.0
recall = tp / true_pairs_total if true_pairs_total else 0.0
f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
missed_by_blocking = true_pairs_total - true_candidate_pairs
true_candidates_not_auto_matched = true_candidate_pairs - tp

print("PAIRWISE AUTOMATIC-MATCH METRICS")
print("--------------------------------")
print(f"True positives (MATCH and same entity) : {tp:,}")
print(f"False positives (MATCH, different entity): {fp:,}")
print(f"False negatives (true pairs not MATCH) : {fn:,}")
print(f"  - True pairs missed by blocking       : {missed_by_blocking:,}")
print(f"  - True candidate pairs not auto-MATCH : {true_candidates_not_auto_matched:,}")
print(f"True pairs marked REVIEW               : {true_review_pairs:,}")
print(f"True pairs marked NO_MATCH             : {true_no_match_pairs:,}")
print(f"Precision                              : {precision * 100:.4f}%")
print(f"Recall (end-to-end pairwise)            : {recall * 100:.4f}%")
print(f"F1-score                               : {f1 * 100:.4f}%")

PAIRWISE AUTOMATIC-MATCH METRICS
--------------------------------
True positives (MATCH and same entity) : 2,241,799
False positives (MATCH, different entity): 7,278
False negatives (true pairs not MATCH) : 266,639
  - True pairs missed by blocking       : 13,121
  - True candidate pairs not auto-MATCH : 253,518
True pairs marked REVIEW               : 158,069
True pairs marked NO_MATCH             : 95,449
Precision                              : 99.6764%
Recall (end-to-end pairwise)            : 89.3703%
F1-score                               : 94.2424%


## 5. Evaluate final Golden Record clusters

This checks the actual `MASTER_CLIENT_ID` clustering, not only direct pair decisions. For pairs, the prediction is that two records belong together when they share a master ID. Pairwise cluster metrics are calculated from group counts, without materializing every cluster pair.

In [0]:
source_map = (
    spark.table(SOURCE_MAP_TABLE)
    .select(
        F.trim(F.col("master_client_id").cast("string")).alias("master_client_id"),
        F.upper(F.trim(F.col("source_system").cast("string"))).alias("source_system"),
        F.trim(F.col("source_id").cast("string")).alias("source_id"),
    )
)

source_map_rows = source_map.count()
source_map_duplicate_keys = (
    source_map.groupBy("source_system", "source_id")
              .count()
              .filter(F.col("count") > 1)
              .limit(1)
              .count()
)
if source_map_duplicate_keys:
    raise ValueError("Gold source map assigns a source record more than once; cluster metrics would be ambiguous.")

cluster_truth = source_map.join(
    truth_silver.select("source_system", "source_id", "entity_key"),
    ["source_system", "source_id"],
    "left",
)
cluster_unmapped_rows = cluster_truth.filter(
    F.col("master_client_id").isNull() | F.col("entity_key").isNull()
).count()
if cluster_unmapped_rows:
    raise ValueError(
        f"{cluster_unmapped_rows:,} source-map rows lack a master ID or ground truth. "
        "Check the serving checkpoint and ground-truth version."
    )
if source_map_rows != eligible_rows:
    raise ValueError(
        f"Gold source-map rows ({source_map_rows:,}) do not equal trusted Silver rows ({eligible_rows:,}). "
        "Do not interpret cluster metrics until coverage is reconciled."
    )

# Total predicted pairs = sum choose(cluster_size, 2).
master_sizes = cluster_truth.groupBy("master_client_id").agg(F.count(F.lit(1)).alias("n"))
predicted_cluster_pairs_row = master_sizes.agg(F.sum(true_pair_expr).alias("pairs")).first()
predicted_cluster_pairs = int(predicted_cluster_pairs_row["pairs"] or 0)

# True-positive cluster pairs share both the same master ID and same entity_key.
master_entity_sizes = cluster_truth.groupBy("master_client_id", "entity_key").agg(F.count(F.lit(1)).alias("n"))
cluster_tp_row = master_entity_sizes.agg(F.sum(true_pair_expr).alias("pairs")).first()
cluster_tp = int(cluster_tp_row["pairs"] or 0)
cluster_fp = predicted_cluster_pairs - cluster_tp
cluster_fn = true_pairs_total - cluster_tp
cluster_precision = cluster_tp / predicted_cluster_pairs if predicted_cluster_pairs else 0.0
cluster_recall = cluster_tp / true_pairs_total if true_pairs_total else 0.0
cluster_f1 = (
    2 * cluster_precision * cluster_recall / (cluster_precision + cluster_recall)
    if (cluster_precision + cluster_recall) else 0.0
)

# Diagnostic counts: a master that contains >1 ground-truth entity is over-merged;
# one ground-truth entity mapped to >1 masters is split / under-merged.
overmerged_master_clusters = (
    cluster_truth.groupBy("master_client_id")
                 .agg(F.countDistinct("entity_key").alias("true_entity_count"))
                 .filter(F.col("true_entity_count") > 1)
                 .count()
)
split_true_entities = (
    cluster_truth.groupBy("entity_key")
                 .agg(F.countDistinct("master_client_id").alias("master_count"))
                 .filter(F.col("master_count") > 1)
                 .count()
)
master_id_count = source_map.select("master_client_id").distinct().count()

print("FINAL CLUSTER METRICS")
print("---------------------")
print(f"Master IDs in source map                 : {master_id_count:,}")
print(f"Predicted same-master record pairs       : {predicted_cluster_pairs:,}")
print(f"Correct same-entity, same-master pairs   : {cluster_tp:,}")
print(f"Incorrect same-master pairs (over-merge) : {cluster_fp:,}")
print(f"Same-entity pairs split across masters   : {cluster_fn:,}")
print(f"Cluster pairwise precision              : {cluster_precision * 100:.4f}%")
print(f"Cluster pairwise recall                 : {cluster_recall * 100:.4f}%")
print(f"Cluster pairwise F1                      : {cluster_f1 * 100:.4f}%")
print(f"Master clusters containing >1 true entity: {overmerged_master_clusters:,}")
print(f"True entities split across >1 master     : {split_true_entities:,}")

FINAL CLUSTER METRICS
---------------------
Master IDs in source map                 : 1,080,592
Predicted same-master record pairs       : 2,361,147
Correct same-entity, same-master pairs   : 2,352,704
Incorrect same-master pairs (over-merge) : 8,443
Same-entity pairs split across masters   : 155,734
Cluster pairwise precision              : 99.6424%
Cluster pairwise recall                 : 93.7916%
Cluster pairwise F1                      : 96.6285%
Master clusters containing >1 true entity: 1,218
True entities split across >1 master     : 82,484


## 6. Compact scorecard and optional export

The scorecard contains aggregate values only; it does not export record-level identifiers or hidden entity keys. If you set `RESULT_OUTPUT_PATH`, write it to a restricted evaluation-only location.

In [0]:
def pct(value):
    return f"{value * 100:.4f}%"

scorecard_rows = [
    ("Population", "Ground-truth mapping rows", f"{gt_rows:,}", "Expected to be the exact frozen FULL ground-truth artifact."),
    ("Population", "Trusted Silver records", f"{silver_rows:,}", "Scoring population after contract and DQ quarantine."),
    ("Population", "Ground-truth rows outside Silver", f"{truth_outside_silver:,}", "Expected quarantined/excluded source records; inspect if unexpected."),
    ("Population", "Eligible underlying entities", f"{eligible_entities:,}", "Distinct hidden entity keys represented in trusted Silver."),
    ("Population", "True record pairs in scoring population", f"{true_pairs_total:,}", "Sum over entities of n*(n-1)/2."),
    ("Blocking", "Candidate pairs", f"{candidate_rows:,}", "Rows in the saved candidate_pairs table."),
    ("Blocking", "True pairs captured", f"{true_candidate_pairs:,}", "Candidate pairs sharing the same hidden entity_key."),
    ("Blocking", "True pairs missed", f"{missed_by_blocking:,}", "True pair denominator minus true candidate pairs."),
    ("Blocking", "Blocking recall", pct(blocking_recall), "True candidate pairs / all eligible true pairs."),
    ("Direct matching", "MATCH true positives", f"{tp:,}", "Decision=MATCH and both records share entity_key."),
    ("Direct matching", "MATCH false positives", f"{fp:,}", "Decision=MATCH but entity_key differs."),
    ("Direct matching", "False negatives", f"{fn:,}", "All eligible true pairs not automatically MATCHed."),
    ("Direct matching", "True pairs in REVIEW", f"{true_review_pairs:,}", "Unresolved true candidates; not counted as automatic TP."),
    ("Direct matching", "True pairs in NO_MATCH", f"{true_no_match_pairs:,}", "True pairs rejected by matching threshold/rules."),
    ("Direct matching", "Precision", pct(precision), "TP / (TP + FP)."),
    ("Direct matching", "End-to-end recall", pct(recall), "TP / all eligible true pairs (including blocking misses)."),
    ("Direct matching", "F1", pct(f1), "Harmonic mean of precision and recall."),
    ("Final clusters", "Master IDs", f"{master_id_count:,}", "Distinct IDs in Gold client_source_map."),
    ("Final clusters", "Over-merged master clusters", f"{overmerged_master_clusters:,}", "Clusters containing multiple hidden entity keys."),
    ("Final clusters", "Split underlying entities", f"{split_true_entities:,}", "Hidden entities mapped to multiple master IDs."),
    ("Final clusters", "Pairwise precision", pct(cluster_precision), "Correct same-master pairs / all same-master pairs."),
    ("Final clusters", "Pairwise recall", pct(cluster_recall), "Correct same-master pairs / all eligible true pairs."),
    ("Final clusters", "Pairwise F1", pct(cluster_f1), "Harmonic mean of cluster pairwise precision and recall."),
]

scorecard = spark.createDataFrame(
    scorecard_rows,
    schema=T.StructType([
        T.StructField("section", T.StringType(), False),
        T.StructField("metric", T.StringType(), False),
        T.StructField("value", T.StringType(), False),
        T.StructField("interpretation", T.StringType(), False),
    ]),
)

display(scorecard)

if RESULT_OUTPUT_PATH:
    (
        scorecard.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", "true")
        .csv(RESULT_OUTPUT_PATH)
    )
    print("Aggregate accuracy scorecard written to the supplied evaluation-only output folder.")
else:
    print("No output path supplied; scorecard is displayed only and has not been saved to storage.")

section,metric,value,interpretation
Population,Ground-truth mapping rows,"2,638,899",Expected to be the exact frozen FULL ground-truth artifact.
Population,Trusted Silver records,"2,638,869",Scoring population after contract and DQ quarantine.
Population,Ground-truth rows outside Silver,30,Expected quarantined/excluded source records; inspect if unexpected.
Population,Eligible underlying entities,"994,420",Distinct hidden entity keys represented in trusted Silver.
Population,True record pairs in scoring population,"2,508,438",Sum over entities of n*(n-1)/2.
Blocking,Candidate pairs,"138,074,372",Rows in the saved candidate_pairs table.
Blocking,True pairs captured,"2,495,317",Candidate pairs sharing the same hidden entity_key.
Blocking,True pairs missed,"13,121",True pair denominator minus true candidate pairs.
Blocking,Blocking recall,99.4769%,True candidate pairs / all eligible true pairs.
Direct matching,MATCH true positives,"2,241,799",Decision=MATCH and both records share entity_key.


No output path supplied; scorecard is displayed only and has not been saved to storage.


## 7. Interpretation and reporting safeguards

- Do not call the candidate decision distribution an accuracy metric.
- Do not use DEV accuracy figures as FULL accuracy.
- Keep direct pairwise metrics separate from final-cluster metrics.
- This measures identity resolution against the synthetic benchmark truth; it does **not** establish factual correctness of Golden Record attributes such as phone, address, or email. That requires an independent, trusted attribute-level reference.
- Review cases are unresolved for automatic matching. If you later measure final post-stewardship results, define a separate evaluation checkpoint and do not mix it with this recorded automatic-decision checkpoint.
- If the run counts, source-map coverage, ground-truth path, or benchmark version do not reconcile, stop and fix the lineage/checkpoint selection before publishing any metric.